# Module 3a — Sorci protocol: the CL-2026-006 **D1** nuisance budget

Reproducible example for `tmc_numerics.modules.sorci`. Regenerates the committed result `results/sorci_nuisance_budget_v0.1.json` from scratch.

Contract: `workplans/toolkit-module3a-sorci-contract-v0.1.md` (D6-locked). Rotating frame w.r.t. the free clock (exact); secular dispersive `H_sec = g_md σ_z n̂` workhorse + full `ω n̂ + g_md σ_z P̂²` validation; `g_md = −ε_m ω_c/4` **derived**.

In [1]:
import json
import numpy as np
import tmc_numerics as tn
from tmc_numerics.modules.sorci import (
    SorciParams, baseline_visibility_check, nuisance_budget, sorci_visibility, _evolve)
print('tmc_numerics', tn.__version__)

tmc_numerics 0.1.0.dev0


## 1. Baseline validation — Sorci Eq. (12) coefficient

Eq. (12) is leading-order in `λ = ε_m ω_c t`; the derived `g_md` is validated by recovering its coefficient: `(1−V)/(λ² sinh² 2r) → 1/16` as `λ→0`.

In [2]:
scaled = SorciParams(omega_c=1e3, omega=1e2, cutoff=80, t=0.1, eps_m=1e-3, r=1.0)
chk = baseline_visibility_check(scaled)
for lam, ratio in sorted(chk['ratios'].items(), reverse=True):
    print(f'  lambda={lam:<6} (1-V)/(lam^2 sinh^2 2r) = {ratio:.5f}')
print('limit ratio =', round(chk['limit_ratio'], 5), ' target 1/16 =', round(chk['target'], 5))
assert abs(chk['limit_ratio'] - 1/16) < 2e-3

  lambda=0.1    (1-V)/(lam^2 sinh^2 2r) = 0.06120
  lambda=0.05   (1-V)/(lam^2 sinh^2 2r) = 0.06217
  lambda=0.025  (1-V)/(lam^2 sinh^2 2r) = 0.06242
limit ratio = 0.06242  target 1/16 = 0.0625


## 2. Latent vs observed nuisance budget (the D1 deliverable)

Marginal-from-signal per channel + full model + interaction residual. **Latent** (`I(C:M)`, `L_state`) is read-out-invariant; **observed** (`L_obs`) is detection-degraded — so the detection row has latent `ΔI = ΔL_state = 0`.

In [3]:
p = SorciParams(omega_c=1e3, omega=1e2, cutoff=80, t=0.1, eps_m=1e-3, r=1.0,
                n_dot=0.5, gamma_phi=0.5, eps_prep=0.05, drive_phase_rate=0.5, eps_det=0.05)
b = nuisance_budget(p)
hdr = f"{'layer':<24}{'channel':<22}{'parameter':<16}{'dI(C:M)':>10}{'dL_state':>11}{'dL_obs':>10}"
print(hdr); print('-'*len(hdr))
print(f"{'Hamiltonian (signal)':<24}{'time-dilation':<22}{'eps_m,r':<16}"
      f"{b.signal[0]:>10.4f}{b.signal[1]:>11.4f}{b.signal[2]:>10.4f}")
for layer, channel, key, val, di, dls, dlo in b.rows:
    print(f'{layer:<24}{channel:<22}{key+"="+str(val):<16}{di:>10.4f}{dls:>11.4f}{dlo:>10.4f}')
print(f"{'-':<24}{'full model':<22}{'(all)':<16}{b.full[0]:>10.4f}{b.full[1]:>11.4f}{b.full[2]:>10.4f}")
print(f"{'-':<24}{'interaction residual':<22}{'':<16}"
      f"{b.interaction_residual[0]:>10.4f}{b.interaction_residual[1]:>11.4f}{b.interaction_residual[2]:>10.4f}")
det = next(r for r in b.rows if r[2] == 'eps_det')
assert det[4] == 0.0 and det[5] == 0.0 and abs(det[6]) > 1e-6
print('\n-> detection row: latent dI = dL_state = 0 (read-out cannot change the state); only dL_obs != 0.')

layer                   channel               parameter          dI(C:M)   dL_state    dL_obs
---------------------------------------------------------------------------------------------
Hamiltonian (signal)    time-dilation         eps_m,r             0.0756     0.0080    0.0080
dynamical-Lindblad      heating               n_dot=0.5          -0.0212     0.0001    0.0001
dynamical-Lindblad      motional dephasing    gamma_phi=0.5      -0.0338    -0.0000   -0.0000
initial-state           prep infidelity       eps_prep=0.05       0.1642     0.0569    0.0569
stochastic-Hamiltonian  drive-phase noise     drive_phase_rate=0.5   -0.0179     0.0484    0.0484
read-out                detection infidelity  eps_det=0.05        0.0000     0.0000    0.0992
-                       full model            (all)               0.1459     0.1106    0.1995
-                       interaction residual                     -0.0211    -0.0028   -0.0130

-> detection row: latent dI = dL_state = 0 (read-out ca

## 3. Extrapolation to the ²⁷Al⁺ operating point (secular workhorse)

The literal point (ε_m=3.3×10⁻¹⁸, ω_c≈2π×1.1×10¹⁵ Hz, t≈1 s) is ~10¹⁵ oscillations; the secular model (n̂ conserved) evaluates it directly at r=2.26.

In [4]:
al = SorciParams(omega_c=2*np.pi*1.1e15, omega=1.0, cutoff=200, t=1.0, eps_m=3.3e-18, r=2.26)
V_al = float(sorci_visibility(_evolve(al, secular=True, include_nuisance=False, npts=30))[-1])
print(f'lambda = eps_m omega_c t = {al.lam:.4e},  r=2.26  ->  V = {V_al:.4f}  (Sorci headline ~ 0.93)')

lambda = eps_m omega_c t = 2.2808e-02,  r=2.26  ->  V = 0.9431  (Sorci headline ~ 0.93)


## 4. Matches the committed artifact

In [5]:
with open('../results/sorci_nuisance_budget_v0.1.json') as f:
    committed = json.load(f)
print('committed model:', committed['model'], '| transfer:', committed['transfer'])
assert abs(committed['al27plus_extrapolation']['V_secular'] - V_al) < 1e-6
assert abs(committed['full_I_Lstate_Lobs'][1] - b.full[1]) < 1e-9
print('OK - notebook regenerates the committed Sorci budget + Al+ extrapolation')

committed model: secular | transfer: {'lambda': 0.1, 'g_md_over_omega': 0.0025, 'omega_t': 10.0}
OK - notebook regenerates the committed Sorci budget + Al+ extrapolation
